In [5]:
NAME: "Cordell Anderson"
EMAIL = "cordellanderson@arizona.edu" 

In [84]:
from typing import Iterator, Tuple, Dict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import RidgeClassifier
import csv
from scipy.sparse import spmatrix

In [75]:
def read_data(file_path: str) -> Iterator[Dict]:
    """
    Reads in the .csv data from the given filepath.
    Each row in the data contains an ID, a sentence, and optionally a label.
    If no label exists in the imported data set, the label will be set to -1.
    
    :param file_path: The path of a .csv data file, formatted as stated above.
    :return: An iterator over dictionaries: where "ID", "sentence", and "label" will be populated.
    """
    output=[]
    first = True
    with open(file_path, mode='r') as file:
        csvFile = csv.reader(file)
        for line in csvFile:
            if first: #skip the first line, it's the headers
                first=False
            else:
                row = {"ID":line[0],"sentence":line[1]}
                if len(line)<3:
                    row["label"]=-1
                else:
                    row["label"]=line[2]
                output.append(row)
    return output

In [119]:
class Classifier:
    def __init__(self):
        """
        Initializes the classifier
        """
        
        self.feature_encoder = TfidfVectorizer(stop_words="english")
        self.classifier = RidgeClassifier(tol=1e-2, solver="sparse_cg")
        self.model = GaussianNB()
        
        
        
    def train(self, data: Iterator[Dict]) -> Tuple[spmatrix,list]:
        """
        Fits the feature_encoder to the provided data.
        Data should be formatted in the same way as the read_data() function formats it.
        
        :param data: The data that will be used to fit the feature_encoder.
        :return: The feature matrix and the sequence of labels associated with it.
        """
        
        documents = []
        labels = []
        for line in data:
            documents.append(line["sentence"])
            labels.append(line["label"])
        feature_matrix = self.feature_encoder.fit_transform(documents)
        self.classifier.fit(feature_matrix, labels)
        return (feature_matrix, labels)
    
        
    def predict(self, data: Iterator[Dict], file_name:str):
        """
        Makes a prediction on the data passed in to the function. Make sure you have run the train() method first!
        Results will be saved to 
        
        :param data: The data that predictions are going to be made on.
        :param file_name: a string containing the file name and path where the predictions will be saved. This
        should end in ".csv".
        """
        documents = []
        ids = []
        for line in data:
            documents.append(line["sentence"])
            ids.append(line["ID"])
        x_test = self.feature_encoder.transform(documents)
        predictions = self.classifier.predict(x_test)
        
        output_data = []
        for i in range(len(ids)):
            output_data.append({"ID":ids[i],"LABEL":predictions[i]})
        
        with open(file_name,'w', newline='') as file:
            headers = ["ID","LABEL"]
            writer = csv.DictWriter(file, fieldnames=headers)
            writer.writeheader()
            writer.writerows(output_data)
        

In [120]:
train_data = read_data("data/train.csv")
test_data = read_data("data/test.csv")
TFIDF = Classifier()
train_x, train_y = TFIDF.train(train_data)


In [121]:
TFIDF.predict(test_data,"output.csv")
